# Setup

In [ ]:
# user setup
import os

# custom Kaggle login (environment variables are recommended)
os.environ["KAGGLE_USERNAME"] = "" or os.getenv("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = "" or os.getenv("KAGGLE_KEY")

# spark configuration
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64" or os.getenv("JAVA_HOME")
os.environ["SPARK_HOME"] = "" or os.getenv("SPARK_HOME")

# subsampling: percentage of commenting users to keep
USERS_PERCENTAGE = 0.1
assert 0 < USERS_PERCENTAGE <= 1, "invalid USER_PERCENTAGE, should be in (0, 1]"

# configuration
MIN_KW_FREQ = 5 # minimum frequency of keywords to be included in the profile (for the dictionary approach)
FEATURES_SIZE = 5000 # number of features, buckets of the hash (for the hashing approach)
LSH_NUM_HASHES = 100 # number of random hyperplanes (signatures/sketches)
LSH_BANDS = 10 # TODO: play with b and r (calculate the threshold for different values)
LSH_ROWS = 10
assert LSH_NUM_HASHES == LSH_BANDS * LSH_ROWS, "invalid LSH_NUM_HASHES, must be equal to LSH_BANDS * LSH_ROWS"


In [ ]:
# setup dataset
%pip install kaggle
!test -d dataset && echo "Skipping dataset download" || (kaggle datasets download -d "benjaminawd/new-york-times-articles-comments-2020" && unzip -d dataset new-york-times-articles-comments-2020.zip && rm -f new-york-times-articles-comments-2020.zip 2> /dev/null)


In [ ]:
# setup spark
!test -d spark && echo "Skipping spark download" || (wget -q https://dlcdn.apache.org/spark/spark-4.1.1/spark-4.1.1-bin-hadoop3.tgz -O spark.tgz && mkdir spark && tar xvf spark.tgz -C spark --strip-components=1 > /dev/null && rm spark.tgz)
%pip install pyspark
%pip install py4j
import pyspark
ss = (pyspark.sql.SparkSession
    .builder
    .getOrCreate())
sc = ss.sparkContext


In [ ]:
csv_options = {
    "header": "true",
    "inferSchema": "true",
    "quote": '"',
    "escape": '"', # quotes " in the dataset are escaped ""
    "multiLine": "true",
    "mode": "PERMISSIVE",
}

full_articles = ss.read.options(**csv_options).csv("dataset/nyt-articles-2020.csv")
full_comments = ss.read.options(**csv_options).csv("dataset/nyt-comments-part0.csv") # TODO: this is a subset of all comments

full_articles.take(1), full_comments.take(1)


([Row(newsdesk='Editorial', section='Opinion', subsection=None, material='Editorial', headline='Protect Veterans From Fraud', abstract='Congress could do much more to protect Americans who have served their country from predatory for-profit colleges.', keywords="['Veterans', 'For-Profit Schools', 'Financial Aid (Education)', 'Frauds and Swindling', 'Colleges and Universities', 'Veterans Affairs Department', 'Federal Trade Commission', 'University of Phoenix', 'Career Education Corporation']", word_count=680, pub_date=datetime.datetime(2020, 1, 1, 0, 18, 54), n_comments=186, uniqueID='nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd')],
 [Row(commentID=104387472, status='approved', commentSequence=104387472, userID=60215558, userDisplayName='magicisnotreal', userLocation='earth', userTitle=None, commentBody='Here is something I think is fraudulent that vets are subject to\n If you use your VA home loan option you have to pay higher interest rates regardless of your credit rating becua

In [ ]:
%pip install mmh3 # non-cryptographic hash, better distribution
# utility hash function: bytes | str -> signed int 32
def h(data):
    import mmh3
    return mmh3.hash(data)


In [ ]:
# parse keywords column utility
def parse_keywords(row):
    import re
    if not row["keywords"]: return []
    kws = re.sub(r'\[|\]|\'|\"', '', row["keywords"].lower()).strip()
    if not kws: return []
    return [w.strip() for w in kws.split(",") if len(w.strip()) > 0]

# keep only relevant information for recommendations porpouse, convert from dataframe to RDD tuples
articles = (full_articles
            .select("uniqueID", "newsdesk", "section", "subsection", "keywords")
            .rdd
            .map(lambda row: (row["uniqueID"], row["newsdesk"], row["section"], row["subsection"], parse_keywords(row))))
comments = (full_comments
            .select("articleID", "userID")
            .rdd
            .map(lambda row: (row["articleID"], row["userID"])))

articles.take(1), comments.take(1)


([('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd',
   'Editorial',
   'Opinion',
   None,
   ['veterans',
    'for-profit schools',
    'financial aid (education)',
    'frauds and swindling',
    'colleges and universities',
    'veterans affairs department',
    'federal trade commission',
    'university of phoenix',
    'career education corporation'])],
 [('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', 60215558)])

## Utility Matrix

In [ ]:
utility_matrix = comments.distinct() # TODO: keep multiple comments on same article or not?
utility_matrix.take(5), utility_matrix.count()


([('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', 60215558),
  ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', 65691034),
  ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', 65110053),
  ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', 66232009),
  ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', 73299044)],
 356283)

## Articles Profile - via dictionary

In [ ]:
# count frequencies
keywords = (articles
            .flatMap(lambda row: [(k, 1) for k in row[4]])
            .reduceByKey(lambda a, b: a + b))

# prune too rare keywords
pruned_keywords = (keywords
                   .filter(lambda x: x[1] >= MIN_KW_FREQ and len(x[0]) <= 100)
                   .map(lambda x: f"KEY {x[0]}")
                   .collect())

# extract all features
features = sorted(pruned_keywords + (articles
            .flatMap(lambda row: [f"{f} {row[i+1].lower().strip()}" for i, f in enumerate(["NEW", "SEC", "SUB"]) if row[i+1]])
            .filter(lambda x: len(x) < 100)
            .distinct()
            .collect()))

# features mapping dictionary, needs to be broadcasted to every node
features_mapping = {f: i for i, f in enumerate(features)}
print(f"Extacted {len(features_mapping)} features (broadcasting at most {((100 * 4) + 4) * len(features_mapping) // 1024} KB)")
broadcast_dict = sc.broadcast(features_mapping)

# build sparse vectors for profiles (article_id, feature_index)
def build_article_profile_dict(row):
    mapping = broadcast_dict.value
    res = []
    for i, f in enumerate(["NEW", "SEC", "SUB"]):
        if not row[i+1]: continue
        f = f"{f} {row[i+1].lower().strip()}"
        if f not in mapping: continue
        res.append((row[0], mapping[f]))
    for k in row[4]:
        k = f"KEY {k.lower().strip()}"
        if k not in mapping: continue
        res.append((row[0], mapping[k]))
    return res

articles_profile_dict = articles.flatMap(build_article_profile_dict)
articles_profile_dict.take(10), articles_profile_dict.count()


Extacted 3555 features (broadcasting at most 1402 KB)


([('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', 3395),
  ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', 3465),
  ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', 3193),
  ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', 1131),
  ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', 1097),
  ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', 1158),
  ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', 628),
  ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', 3194),
  ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', 1086),
  ('nyt://article/9edddb54-0aa3-5835-a833-d311a76f1e7c', 3399)],
 158079)

## Articles Profile - via hashing

In [ ]:
# build sparse vectors for profiles (article_id, feature_hash)
def build_article_profile_hash(row):
    res = []
    for i, f in enumerate(["NEW", "SEC", "SUB"]):
        if not row[i+1]: continue
        f = f"{f} {row[i+1].lower().strip()}"
        res.append((row[0], h(f) % FEATURES_SIZE))
    for k in row[4]:
        k = f"KEY {k.lower().strip()}"
        res.append((row[0], h(k) % FEATURES_SIZE))
    return res

articles_profile_hash = articles.flatMap(build_article_profile_hash)
articles_profile_hash.take(10), articles_profile_hash.count()


([('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', 653),
  ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', 2704),
  ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', 2692),
  ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', 1483),
  ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', 4497),
  ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', 260),
  ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', 1675),
  ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', 2683),
  ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', 3196),
  ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', 598)],
 185396)

### WARNING

From now on, the operations will be **independent** of how the profiles were computed (via dictionary or hashing).
Change the content of next cell to change approach: (`articles_profile_dict` or `articles_profile_hash`)

In [ ]:
articles_profile = articles_profile_hash


## Users Profile

In [ ]:
# build user profiles by joining utility matrix with article profiles, then counting feature frequencies for each user
users_profile = (utility_matrix
                     .join(articles_profile)
                     .map(lambda x: ((x[1][0], x[1][1]), 1))
                     .reduceByKey(lambda a, b: a + b))
users_profile.take(10), users_profile.count()


([((5646097, 535), 36),
  ((5646097, 2881), 36),
  ((5646097, 4221), 36),
  ((86273619, 535), 11),
  ((86273619, 2881), 11),
  ((86273619, 4221), 11),
  ((61536727, 535), 9),
  ((61536727, 2881), 9),
  ((61536727, 4221), 9),
  ((41508509, 535), 20)],
 2801463)

## Computing Similarity - LSH using hyperplanes (sketches)

In [25]:
import random

# random vectors of +1 and -1 of dimensione FEATURES_SIZE
random_hyperplanes = [[random.choice([-1, 1]) for _ in range(FEATURES_SIZE)] for _ in range(LSH_NUM_HASHES)]
b_hyperplanes = sc.broadcast(random_hyperplanes)

def compute_sketch(row):
    item_id, sparse_features = row
    sketch = []
    for plane in b_hyperplanes.value:
        # dot product: if above or below the hyperplane
        # only considering the features that exist in the sparse vector
        dot_product = sum(weight * plane[feat] for feat, weight in sparse_features)
        # 1 if positive, 0 if negative
        sketch.append(1 if dot_product > 0 else 0)
    return (item_id, sketch)

# compute sketches ("equivalent" of signature) for users and articles
users_sketches = (users_profile
                  .map(lambda x: (x[0][0], ((x[0][1], x[1]))))
                  .groupByKey()
                  .mapValues(list)
                  .map(compute_sketch))
articles_sketches = (articles_profile
                  .map(lambda x: (x[0], ((x[1], 1))))
                  .groupByKey()
                  .mapValues(list)
                  .map(compute_sketch))

users_sketches.take(1), articles_sketches.take(1)


([(13120442,
   [1,
    0,
    1,
    0,
    1,
    0,
    1,
    0,
    0,
    0,
    0,
    1,
    1,
    1,
    1,
    1,
    0,
    0,
    1,
    0,
    1,
    0,
    1,
    0,
    0,
    1,
    1,
    0,
    0,
    1,
    1,
    0,
    0,
    1,
    0,
    0,
    1,
    0,
    1,
    1,
    1,
    0,
    1,
    0,
    0,
    1,
    1,
    0,
    0,
    0,
    1,
    1,
    1,
    1,
    1,
    0,
    1,
    0,
    1,
    0,
    0,
    1,
    0,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    0,
    1,
    0,
    0,
    1,
    1,
    0,
    1,
    0,
    1,
    0,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    0,
    0,
    1,
    1,
    0,
    0,
    1,
    0,
    0,
    0])],
 [('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd',
   [1,
    0,
    0,
    0,
    0,
    1,
    1,
    1,
    1,
    0,
    1,
    1,
    0,
    1,
    1,
    1,
    0,
    1,
    0,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    0,
    1,
    0,
    0,
    0,
    1,
   

In [26]:
def lsh_buckets(row, what):
    item_id, sketch = row
    res = []
    for band in range(LSH_BANDS):
        band_signature = tuple(sketch[band*LSH_ROWS : (band+1)*LSH_ROWS])
        bucket_hash = h(f"b{band}_{band_signature}") # include band index to avoid collisions between different bands
        res.append((bucket_hash, (what, item_id)))
    return res

users_buckets = users_sketches.flatMap(lambda x: lsh_buckets(x, "USER"))
articles_buckets = articles_sketches.flatMap(lambda x: lsh_buckets(x, "ARTICLE"))

users_buckets.take(10), articles_buckets.take(10)


([(-1528755988, ('USER', 13120442)),
  (573874805, ('USER', 13120442)),
  (109712757, ('USER', 13120442)),
  (456923391, ('USER', 13120442)),
  (-686120117, ('USER', 13120442)),
  (-2031674836, ('USER', 13120442)),
  (1117070857, ('USER', 13120442)),
  (765610745, ('USER', 13120442)),
  (464771685, ('USER', 13120442)),
  (-961146683, ('USER', 13120442))],
 [(1272973056,
   ('ARTICLE', 'nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd')),
  (1941486667,
   ('ARTICLE', 'nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd')),
  (1521742613,
   ('ARTICLE', 'nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd')),
  (1725585274,
   ('ARTICLE', 'nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd')),
  (1266562358,
   ('ARTICLE', 'nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd')),
  (-1947116141,
   ('ARTICLE', 'nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd')),
  (764242108,
   ('ARTICLE', 'nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd')),
  (928440656,
   ('ARTICLE', 'nyt://arti

In [ ]:
# no distinct 20 bands x 5 rows = ~1b results in 3min 20s
# no distinct 10 bands x 10 rows = ~50m results in 30s
#    distinct 10 bands x 10 rows = ~30m results in 6min 50s

recommendations = (users_buckets
                   .join(articles_buckets)
                   .map(lambda x: (x[1][0][1], x[1][1][1])))
                   # TODO: distinct and subtrack make sense but are really really slow and use a loooot of memory
                   # right now the distinct and subtraction are done per-user in the actual recommendation step, but it would be better to do them here
                   # .distinct()
                   # .subtract(utility_matrix))

recommendations.take(10), recommendations.count()


([(13120442, 'nyt://article/7a227b82-431e-5bc5-a0c0-7d9581e89cc8'),
  (13120442, 'nyt://article/f9183d03-2495-5377-a396-d0efbdfbf764'),
  (13120442, 'nyt://article/2e747dcd-f037-53e5-a219-a2bec2b77003'),
  (13120442, 'nyt://article/63ebfdab-87ee-5d8d-87bd-3e4a44325ec3'),
  (13120442, 'nyt://article/75e15e35-7d18-57fa-80ab-64e1926fdbd2'),
  (13120442, 'nyt://article/65cff85d-1be4-5453-87f8-07049b4d2514'),
  (13120442, 'nyt://article/ca5694f3-c121-53e3-9a35-1f429f23daa8'),
  (13120442, 'nyt://article/b1249e4c-5ead-54d6-b076-cb4c009159b4'),
  (13120442, 'nyt://article/34be1c27-d0a7-50d7-bf8e-1fe3925351b7'),
  (13120442, 'nyt://article/b55374dc-5333-52c3-a6bb-cbe48d44298c')],
 59044038)

In [43]:
USER_ID = 13120442 # 0.24054 over 377 recommendations ['top 5: 0.5834', 'top 10: 0.56436', 'top 20: 0.54242']
# USER_ID = 86273619 # 0.61674 over 454 recommendations ['top 5: 0.98126', 'top 10: 0.98126', 'top 20: 0.98126']
# USER_ID = 61536727 # 0.61831 over 457 recommendations ['top 5: 0.98198', 'top 10: 0.98198', 'top 20: 0.98198']
# USER_ID = 79217680 # 0.03767 over 178 recommendations ['top 5: 0.6175', 'top 10: 0.47482', 'top 20: 0.32519']
# USER_ID = 77701784 # 0.12554 over 355 recommendations ['top 5: 0.48529', 'top 10: 0.44943', 'top 20: 0.38532']

# cosine similarity between sparse profiles, user_profile and article_profile should be dicts {feature_index: weight}
def cosine_similarity(user_profile, article_profile):
    dot_product = sum(user_profile.get(feat, 0) * weight for feat, weight in article_profile.items())
    user_norm = sum(weight ** 2 for weight in user_profile.values()) ** 0.5
    article_norm = sum(weight ** 2 for weight in article_profile.values()) ** 0.5
    if user_norm == 0 or article_norm == 0:
        return 0.0
    return dot_product / (user_norm * article_norm)

# single user recommendations with article profile
user_reccs = (recommendations
            .filter(lambda x: x[0] == USER_ID)
            # TODO: next two steps should be done above in the recommendations RDD
            .distinct()
            .subtract(utility_matrix.filter(lambda x: x[1] == USER_ID).map(lambda x: (x[1], x[0]))) # remove already commented articles
            .map(lambda x: (x[1], None)) # without a valid pair, the join will not return anything
            .join(articles_profile.map(lambda x: (x[0], (x[1], 1))).groupByKey().mapValues(list))
            .map(lambda x: (x[0], x[1][1])))

user_profile = dict(users_profile
                    .filter(lambda x: x[0][0] == USER_ID)
                    .map(lambda x: (x[0][1], x[1]))
                    .collect())

similarities = []
for article, profile in user_reccs.collect():
    sim = cosine_similarity(user_profile, dict(profile))
    similarities.append(round(sim, 5))
    # print(article, sim)

similarities.sort(reverse=True)
print(f"Average similarity: {(sum(similarities) / len(similarities)):.5f} over {len(similarities)} recommendations {[('top ' + str(qty) + ': ' + str(round(sum(similarities[:qty]) / qty, 5))) for qty in [5, 10, 20]]}")


Average similarity: 0.24506 over 384 recommendations ['top 5: 0.60967', 'top 10: 0.58667', 'top 20: 0.56074']


---
---

In [44]:
# setup pandas (tests before scaling to spark)
%pip install pandas
import pandas as pd
articles = pd.read_csv("dataset/nyt-articles-2020.csv")
comments = pd.read_csv("dataset/nyt-comments-part0.csv") # TODO: this is already a sample
print(f"Dataset size: {len(articles)=}, {len(comments)=}")
(articles.columns, comments.columns)


Note: you may need to restart the kernel to use updated packages.
Dataset size: len(articles)=16787, len(comments)=500000


(Index(['newsdesk', 'section', 'subsection', 'material', 'headline', 'abstract',
        'keywords', 'word_count', 'pub_date', 'n_comments', 'uniqueID'],
       dtype='object'),
 Index(['commentID', 'status', 'commentSequence', 'userID', 'userDisplayName',
        'userLocation', 'userTitle', 'commentBody', 'createDate', 'updateDate',
        'approveDate', 'recommendations', 'replyCount', 'editorsSelection',
        'parentID', 'parentUserDisplayName', 'depth', 'commentType', 'trusted',
        'recommendedFlag', 'permID', 'isAnonymous', 'articleID'],
       dtype='object'))

In [45]:
# subsampling
# WARNING: executing this cell multiple times without executing previous cell will continue to shrink the dataset
if USERS_PERCENTAGE < 1:
    total_users = len(comments['userID'].unique())
    subsampled_users = int(total_users * USERS_PERCENTAGE)
    top_users = comments['userID'].value_counts().nlargest(subsampled_users).index # keep top users
    print(top_users) # TODO

    print(f"Subsampling from {total_users} to {subsampled_users} users:")

    print(f"- comments: {len(comments)} -> ", end="")
    comments = comments[comments['userID'].isin(top_users)].copy() # keep only comments from these users
    print(f"{len(comments)}")

    active_article_ids = comments['articleID'].unique() # keep only commented articles
    print(f"- articles: {len(articles)} -> ", end="")
    articles = articles[articles['uniqueID'].isin(active_article_ids)].copy() # keep only commented articles
    print(f"{len(articles)}")


Index([ 67892453,  68938663,  73444633,  33668404,  69498476,  78358680,
        63483890,  72967915,  87081404,  93082167,
       ...
        49632588,     61360,  80245071,  22822988,  28571829,  21227474,
        65684325, 103084739,  78118899,  42471841],
      dtype='int64', name='userID', length=9143)
Subsampling from 91437 to 9143 users:
- comments: 500000 -> 313145
- articles: 16787 -> 1593


# Utility Matrix

In [46]:
utility_matrix = pd.crosstab(index=comments['userID'], columns=comments['articleID'])
utility_matrix = (utility_matrix > 0).astype(int) # ignore multiple comments on the same article
utility_matrix


articleID,nyt://article/00036db7-8494-5141-b0ac-414118caabee,nyt://article/00168469-d42e-5a6f-9c81-90e3306c6c85,nyt://article/004716c5-8943-5bf4-8293-864ab083d676,nyt://article/00573cb4-92bb-53b7-ad9a-6d4d1e734704,nyt://article/00bea7d5-c04a-509d-a82f-7537a16dfeb1,nyt://article/012aeb93-8149-522b-a18b-d6cfc620fb22,nyt://article/016cf3b3-56e9-53e4-882c-549209211afb,nyt://article/0178a7cd-b6b8-58b2-afe4-3d1f68aef248,nyt://article/01a433cd-37ce-5e77-a39b-fe4af162a34f,nyt://article/01b7c5aa-9cf5-56a0-8162-3b8d606ce533,...,nyt://interactive/c46aa52e-ec78-5e20-8145-acaa9d1d5705,nyt://interactive/c696fd2f-2287-562e-91ec-3f07747a9923,nyt://interactive/c6ec5381-c3f8-5a71-b23a-a0d9b18c7237,nyt://interactive/c891ed57-1184-52d9-bd9c-cf9f7bf23f13,nyt://interactive/d0598a7b-06ba-56db-9df4-7a2afa1c95e3,nyt://interactive/dcc11686-2ae2-562c-9e2b-738e9140b9d4,nyt://interactive/df6cd908-52b0-5851-ac81-e7b8039d50d1,nyt://interactive/e31c6ca3-4bea-5a56-b358-7986e7590f1d,nyt://interactive/e5bd1257-4e08-5656-bc03-091a2597e336,nyt://interactive/f91ad36a-9e5c-5e95-a210-37f1b797fdf3
userID,,,,,,,,,,,,,,,,,,,,,
2593,0,0,0,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
8010,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
8638,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
13323,0,0,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
17282,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
109551783,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
109587542,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
109738098,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


# Content-Based Recommendations

### Articles Profile

In [47]:
# items profile
feature_columns = ["newsdesk", "section", "subsection"]
item_features_matrix = pd.get_dummies(articles[feature_columns], dtype=int)
item_features_matrix.index = articles['uniqueID'].values
# articles["profile"] = item_features_matrix.values.tolist()
item_features_matrix


,newsdesk_Arts,newsdesk_Arts&Leisure,newsdesk_BookReview,newsdesk_Books,newsdesk_Business,newsdesk_Climate,newsdesk_Culture,newsdesk_Dining,newsdesk_Editorial,newsdesk_Express,...,subsection_Pro Football,subsection_Self-Care,subsection_Skiing,subsection_Soccer,subsection_Sunday Review,subsection_Television,subsection_Tennis,subsection_The Daily,subsection_Weddings,"subsection_Wine, Beer & Cocktails"
nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd,0,0,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
nyt://article/9edddb54-0aa3-5835-a833-d311a76f1e7c,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
nyt://article/04bc90f0-b20b-511c-b5bb-3ce13194163f,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
nyt://article/bd8647b3-8ec6-50aa-95cf-2b81ed12d2dd,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
nyt://article/ac742403-9ccd-522f-9a1e-a90feb6c0acd,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
nyt://article/b9c293a0-dfb4-5630-9948-8a39be8afce8,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
nyt://article/203dd447-5d08-5745-95b8-b97de174d4b9,0,0,0,0,0,0,0,0,0,0,...,0,0,1,0,0,0,0,0,0,0
nyt://article/1e2029bd-8b3d-5e1e-9e78-28e5ff746998,0,0,0,0,0,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,0
nyt://article/697c75a8-2600-5e81-98ac-38193d89ae12,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0


### Users Profile

In [48]:
# user profile: count how many times each user commented on articles with each label
user_profiles = []

for user_id in utility_matrix.index:
    commented_articles = utility_matrix.loc[user_id][utility_matrix.loc[user_id] == 1].index # articles commented by this user
    user_profile = item_features_matrix.loc[item_features_matrix.index.isin(commented_articles)].sum(axis=0) # sum features vectors for these articles
    user_profiles.append(user_profile)

user_features_matrix = pd.DataFrame(user_profiles, index=utility_matrix.index)
user_features_matrix


,newsdesk_Arts,newsdesk_Arts&Leisure,newsdesk_BookReview,newsdesk_Books,newsdesk_Business,newsdesk_Climate,newsdesk_Culture,newsdesk_Dining,newsdesk_Editorial,newsdesk_Express,...,subsection_Pro Football,subsection_Self-Care,subsection_Skiing,subsection_Soccer,subsection_Sunday Review,subsection_Television,subsection_Tennis,subsection_The Daily,subsection_Weddings,"subsection_Wine, Beer & Cocktails"
userID,,,,,,,,,,,,,,,,,,,,,
2593,0,0,0,0,2,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
8010,0,0,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
8638,0,0,0,0,0,0,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0
13323,0,0,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
17282,0,2,0,0,3,0,1,0,2,0,...,2,0,0,0,3,2,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
109551783,0,0,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
109587542,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
109738098,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


### Calculate Recommendations

In [49]:
from sklearn.metrics.pairwise import cosine_similarity # TODO: implement from scratch?
similarity_array = cosine_similarity(user_features_matrix, item_features_matrix)
similarity_df = pd.DataFrame(
    similarity_array,
    index=user_features_matrix.index,
    columns=item_features_matrix.index
)
similarity_df.to_csv("similarity_df.csv")
similarity_df


,nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd,nyt://article/9edddb54-0aa3-5835-a833-d311a76f1e7c,nyt://article/04bc90f0-b20b-511c-b5bb-3ce13194163f,nyt://article/bd8647b3-8ec6-50aa-95cf-2b81ed12d2dd,nyt://article/ac742403-9ccd-522f-9a1e-a90feb6c0acd,nyt://article/dea10063-7440-586f-bc32-db1dbaa9893f,nyt://article/dfd5bb59-f330-5c01-ae02-221e6f9cae7a,nyt://article/6227c991-6a3d-5eb1-92c9-a618f6e20dbd,nyt://article/7d9a4f78-44ef-5756-bc2b-4c13cbfa0d5a,nyt://article/da74e3b3-f027-5b03-9052-0bc2e69a2f3e,...,nyt://article/23ec0066-2831-599d-bbff-cef34498ab07,nyt://article/2ae8b3b1-f038-5953-a83a-8b75389010da,nyt://article/67b90b5b-fbb1-5f4b-ab12-7fcc24793bcf,nyt://article/d118dda5-b976-56f9-81c7-a961dcff8e84,nyt://article/46182ac4-985b-5b2d-9c4c-5ca4c9552fdd,nyt://article/b9c293a0-dfb4-5630-9948-8a39be8afce8,nyt://article/203dd447-5d08-5745-95b8-b97de174d4b9,nyt://article/1e2029bd-8b3d-5e1e-9e78-28e5ff746998,nyt://article/697c75a8-2600-5e81-98ac-38193d89ae12,nyt://article/49cedd40-d966-5fa1-a04a-1db9abc2756a
userID,,,,,,,,,,,,,,,,,,,,,
2593,0.185341,0.0,0.030890,0.030890,0.680985,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.00000,0.0,0.000000,0.226995,0.370681,0.370681,0.000000,0.000000,0.302660,0.403547
8010,0.046424,0.0,0.000000,0.000000,0.000000,0.278543,0.278543,0.278543,0.000000,0.000000,...,0.00000,0.0,0.000000,0.644383,0.092848,0.092848,0.000000,0.000000,0.075810,0.947623
8638,0.396059,0.0,0.099015,0.099015,0.080845,0.000000,0.000000,0.000000,0.099015,0.161690,...,0.19803,0.0,0.099015,0.242536,0.693103,0.693103,0.000000,0.000000,0.565916,0.161690
13323,0.553519,0.0,0.000000,0.000000,0.150649,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.00000,0.0,0.000000,0.075324,0.968658,0.968658,0.000000,0.000000,0.790906,0.150649
17282,0.552536,0.0,0.000000,0.000000,0.000000,0.157867,0.157867,0.157867,0.039467,0.000000,...,0.00000,0.0,0.118401,0.193347,0.868271,0.868271,0.128898,0.193347,0.805614,0.128898
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
109551783,0.000000,0.0,0.250000,0.250000,0.408248,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.00000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
109587542,0.267261,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.145479,...,0.00000,0.0,0.000000,0.727393,0.534522,0.534522,0.000000,0.000000,0.436436,0.727393
109738098,0.500000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.00000,0.0,0.000000,0.000000,1.000000,1.000000,0.000000,0.000000,0.816497,0.000000


In [50]:
articles_lookup = articles.set_index("uniqueID", drop=False)

def print_article(i, article_id, score=None):
    if article_id in articles_lookup.index:
        row = articles_lookup.loc[article_id]
        title = row.get("headline", row.get("title", "N/A"))
        abstract = row.get("abstract", "N/A")
        newsdesk = row.get("newsdesk", "N/A")
        section = row.get("section", "N/A")
        subsection = row.get("subsection", "N/A")
    else:
        title = abstract = newsdesk = section = subsection = "N/A"

    score_str = f" | score: {score:.5f}" if score is not None else ""
    print(f"{i}. article_id: {article_id}{score_str} | {title} | {abstract} | {newsdesk=} | {section=} | {subsection=}")

def print_results(user_id, recommendations=5):

    commented_ids = []
    if user_id in utility_matrix.index:
        commented = utility_matrix.loc[user_id]
        commented_ids = commented[commented > 0].index.tolist()

    scores = similarity_df.loc[user_id].sort_values(ascending=False)
    # remove alreayd commented articles
    scores = scores.drop(index=commented_ids, errors="ignore")


    print(f"=== Top {len(scores.head(recommendations))} recommendations for user {user_id}:")
    for i, (article_id, score) in enumerate(scores.head(recommendations).items(), start=1):
        print_article(i, article_id, score)

    print(f"\n=== Articles already commented by user {user_id} ({len(commented_ids)}):")
    for i, article_id in enumerate(commented_ids, start=1):
        print_article(i, article_id)

print_results(similarity_df.index[12], 50)


=== Top 50 recommendations for user 42569:
1. article_id: nyt://article/8a86511a-17d6-55e1-aa07-7866cd8ef600 | score: 0.75000 | Trump Acts Like a Politician. That’s Not an Impeachable Offense. | Receiving a “political benefit” does not transform an otherwise legal action (like requesting an investigation) into an abuse of power. | newsdesk='OpEd' | section='Opinion' | subsection=nan
2. article_id: nyt://article/9279bb5d-2909-5a7e-a986-4c962749b224 | score: 0.75000 | An Open Letter to John Lewis | As you fight a new battle, we owe you a debt of gratitude for being a moral compass for our nation. | newsdesk='OpEd' | section='Opinion' | subsection=nan
3. article_id: nyt://article/2d63ffca-75d3-5e97-b794-42cc57fa377b | score: 0.75000 | Trump Has a Bizarre Idea of Winning | Let’s tally up the results of his efforts with Iran. | newsdesk='OpEd' | section='Opinion' | subsection=nan
4. article_id: nyt://article/17e3f286-50bc-5127-af57-164567d1a380 | score: 0.75000 | The Many Polarizations of A